In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
PROJECT_ID = "claude-project-511805"
client = bigquery.Client(project=PROJECT_ID)

df = client.query("SELECT * FROM `ga4_funnel.model_first_session`").to_dataframe()
print(df.shape)
print("Positive rate:", round(df["purchased_later"].mean(), 4))

(177444, 17)
Positive rate: 0.011


In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split

target = "purchased_later"
num_cols = ["fs_page_views", "fs_item_views", "fs_add_to_cart", "fs_searches",
            "fs_promo_views", "fs_scrolls", "fs_began_checkout", "fs_engaged_seconds"]
top_countries = df["country"].value_counts().index[:10]
df["country_grp"] = df["country"].where(df["country"].isin(top_countries), "Other")
cat_cols = ["device", "os", "channel", "country_grp", "fs_hour", "fs_dow"]
df[cat_cols] = df[cat_cols].fillna("Unknown").astype(str)

X, y = df[num_cols + cat_cols], df[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

pre = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])

logreg = Pipeline([("pre", pre),
                   ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))])
logreg.fit(X_train, y_train)
p_lr = logreg.predict_proba(X_test)[:, 1]

print("Random baseline PR-AUC:", round(y_test.mean(), 4))
print("LogReg PR-AUC:", round(average_precision_score(y_test, p_lr), 4))
print("LogReg ROC-AUC:", round(roc_auc_score(y_test, p_lr), 4))

Random baseline PR-AUC: 0.011
LogReg PR-AUC: 0.0963
LogReg ROC-AUC: 0.8419


In [ ]:
res = pd.DataFrame({"y": y_test.values, "p": p_lr}).sort_values("p", ascending=False)
base = res["y"].mean()
for pct in [1, 5, 10, 20]:
    top = res.head(int(len(res) * pct / 100))
    print(f"Top {pct:>2}%: precision {top['y'].mean():.3f} | "
          f"captures {top['y'].sum() / res['y'].sum():.1%} of later buyers | "
          f"lift {top['y'].mean() / base:.1f}x")

Top  1%: precision 0.167 | captures 15.1% of later buyers | lift 15.2x
Top  5%: precision 0.093 | captures 42.1% of later buyers | lift 8.4x
Top 10%: precision 0.063 | captures 57.3% of later buyers | lift 5.7x
Top 20%: precision 0.040 | captures 72.6% of later buyers | lift 3.6x


In [ ]:
from xgboost import XGBClassifier
from sklearn.base import clone

spw = (y_train == 0).sum() / (y_train == 1).sum()
xgb = Pipeline([("pre", clone(pre)),
                ("model", XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05,
                                        subsample=0.8, colsample_bytree=0.8,
                                        scale_pos_weight=spw, eval_metric="aucpr",
                                        random_state=42, n_jobs=-1))])
xgb.fit(X_train, y_train)
p_xgb = xgb.predict_proba(X_test)[:, 1]
print("XGB PR-AUC:", round(average_precision_score(y_test, p_xgb), 4))
print("XGB ROC-AUC:", round(roc_auc_score(y_test, p_xgb), 4))

XGB PR-AUC: 0.1032
XGB ROC-AUC: 0.8741


In [ ]:
def lift_table(y_true, p, name):
    res = pd.DataFrame({"y": np.asarray(y_true), "p": p}).sort_values("p", ascending=False)
    base, rows = res["y"].mean(), []
    for pct in [1, 5, 10, 20]:
        top = res.head(int(len(res) * pct / 100))
        rows.append({"model": name, "top_pct": pct,
                     "precision": round(top["y"].mean(), 3),
                     "capture":   round(top["y"].sum() / res["y"].sum(), 3),
                     "lift":      round(top["y"].mean() / base, 1)})
    return pd.DataFrame(rows)

print(pd.concat([lift_table(y_test, p_lr,  "LogReg"),
                 lift_table(y_test, p_xgb, "XGBoost")]).to_string(index=False))

  model  top_pct  precision  capture  lift
 LogReg        1      0.167    0.151  15.2
 LogReg        5      0.093    0.421   8.4
 LogReg       10      0.063    0.573   5.7
 LogReg       20      0.040    0.726   3.6
XGBoost        1      0.167    0.151  15.2
XGBoost        5      0.098    0.446   8.9
XGBoost       10      0.068    0.613   6.1
XGBoost       20      0.043    0.777   3.9


In [ ]:
from sklearn.inspection import permutation_importance

best = xgb   # change to logreg if LogReg wins
imp = permutation_importance(best, X_test, y_test, scoring="average_precision",
                             n_repeats=5, random_state=42, n_jobs=-1)
print(pd.Series(imp.importances_mean, index=X_test.columns)
        .sort_values(ascending=False).round(4))

fs_engaged_seconds    0.0747
fs_item_views         0.0551
fs_scrolls            0.0469
fs_page_views         0.0328
fs_promo_views        0.0232
fs_dow                0.0028
fs_add_to_cart        0.0010
fs_searches           0.0010
fs_hour               0.0009
country_grp           0.0006
device                0.0003
channel              -0.0008
os                   -0.0012
fs_began_checkout    -0.0027
dtype: float64


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in [("LogReg", logreg), ("XGBoost", xgb)]:
    s = cross_val_score(model, X_train, y_train, cv=cv, scoring="average_precision")
    print(f"{name}: PR-AUC {s.mean():.4f} ± {s.std():.4f} | folds {np.round(s, 4)}")

LogReg: PR-AUC 0.0906 ± 0.0054 | folds [0.0907 0.092  0.0939 0.096  0.0805]
XGBoost: PR-AUC 0.1064 ± 0.0118 | folds [0.098  0.1024 0.1032 0.1296 0.0986]


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_dist = {
    "model__n_estimators":     randint(200, 800),
    "model__max_depth":        randint(3, 8),
    "model__learning_rate":    uniform(0.01, 0.15),
    "model__min_child_weight": randint(1, 20),
    "model__subsample":        uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.6, 0.4),
}
search = RandomizedSearchCV(
    xgb, param_dist, n_iter=20, scoring="average_precision",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    random_state=42, verbose=1)
search.fit(X_train, y_train)
print("Best CV PR-AUC:", round(search.best_score_, 4))
print(search.best_params_)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best CV PR-AUC: 0.1091
{'model__colsample_bytree': np.float64(0.7433862914177091), 'model__learning_rate': np.float64(0.027380358928769458), 'model__max_depth': 3, 'model__min_child_weight': 8, 'model__n_estimators': 232, 'model__subsample': np.float64(0.6254233401144095)}


In [ ]:
final = search.best_estimator_   # or: final = logreg   (if Cell 8 was a tie)
p_final = final.predict_proba(X_test)[:, 1]
print("Final test PR-AUC:", round(average_precision_score(y_test, p_final), 4),
      "| baseline:", round(y_test.mean(), 4))
print(lift_table(y_test, p_final, "Final").to_string(index=False))

Final test PR-AUC: 0.1112 | baseline: 0.011
model  top_pct  precision  capture  lift
Final        1      0.194    0.176  17.6
Final        5      0.102    0.464   9.3
Final       10      0.071    0.640   6.4
Final       20      0.043    0.783   3.9


In [ ]:
from sklearn.model_selection import cross_val_predict

final_model = clone(search.best_estimator_)
oof = cross_val_predict(final_model, X, y,
                        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                        method="predict_proba")[:, 1]

scores = df[["user_pseudo_id", "channel", "device", "country_grp", "first_date"]].copy()
scores["propensity"] = oof
scores["decile"] = 10 - pd.qcut(scores["propensity"].rank(method="first"), 10, labels=False)
scores["purchased_later"] = y.values

client.load_table_from_dataframe(
    scores, f"{PROJECT_ID}.ga4_funnel.propensity_scores",
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")).result()
print("Uploaded rows:", len(scores))

Uploaded rows: 177444
